# Bronze to Silver - Northstar Transactions

This notebook reads the raw `transactions_raw` Delta table from the **bronze** Lakehouse, cleans and deduplicates transaction rows, and writes a curated `transactions_clean` Delta table to the **silver** Lakehouse.

Before running:
1. Attach the `bronze` and `silver` Lakehouses to this notebook.
2. Confirm Lab 1 created `bronze.transactions_raw`.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


## Configure Lakehouse names

Update these values only if your Lakehouses use different names.


In [ ]:
bronze_lakehouse = "bronze"
silver_lakehouse = "silver"
raw_table = "transactions_raw"
silver_table = "transactions_clean"


## Read raw bronze data


In [ ]:
raw_df = spark.table(f"{bronze_lakehouse}.{raw_table}")

raw_count = raw_df.count()
print(f"Raw rows: {raw_count:,}")
display(raw_df.limit(10))


## Standardize data types and add useful derived columns


In [ ]:
clean_base = (
    raw_df
    .withColumn("TransactionID", F.trim(F.col("TransactionID").cast("string")))
    .withColumn("CardToken", F.when(F.trim(F.col("CardToken").cast("string")) == "", None).otherwise(F.trim(F.col("CardToken").cast("string"))))
    .withColumn("MerchantID", F.trim(F.col("MerchantID").cast("string")))
    .withColumn("MerchantCategory", F.trim(F.col("MerchantCategory").cast("string")))
    .withColumn("Region", F.trim(F.col("Region").cast("string")))
    .withColumn("Country", F.trim(F.col("Country").cast("string")))
    .withColumn("Amount", F.col("Amount").cast("decimal(18,2)"))
    .withColumn("Currency", F.upper(F.trim(F.col("Currency").cast("string"))))
    .withColumn("AuthStatus", F.upper(F.trim(F.col("AuthStatus").cast("string"))))
    .withColumn("Timestamp", F.to_timestamp(F.col("Timestamp")))
    .withColumn("IsInternational", F.col("IsInternational").cast("boolean"))
    .withColumn("ProcessingFee", F.col("ProcessingFee").cast("decimal(18,4)"))
    .withColumn("TransactionDate", F.to_date(F.col("Timestamp")))
    .withColumn("TransactionYear", F.year(F.col("Timestamp")))
    .withColumn("TransactionMonth", F.month(F.col("Timestamp")))
    .withColumn("TransactionDay", F.dayofmonth(F.col("Timestamp")))
    .withColumn("HighValueFlag", F.when(F.col("Amount") >= F.lit(10000), F.lit(True)).otherwise(F.lit(False)))
    .withColumn(
        "AmountBand",
        F.when(F.col("Amount") < 100, "Under 100")
         .when(F.col("Amount") < 1000, "100-999")
         .when(F.col("Amount") < 10000, "1,000-9,999")
         .otherwise("10,000+")
    )
)


## Remove rows without transaction IDs and deduplicate

For duplicate `TransactionID` values, keep the latest row by `Timestamp`.


In [ ]:
window = Window.partitionBy("TransactionID").orderBy(F.col("Timestamp").desc_nulls_last())

silver_df = (
    clean_base
    .filter(F.col("TransactionID").isNotNull())
    .withColumn("rn", F.row_number().over(window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

silver_count = silver_df.count()
print(f"Raw rows: {raw_count:,}")
print(f"Silver rows after dedupe: {silver_count:,}")
display(silver_df.limit(10))


## Write the silver Delta table


In [ ]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_lakehouse}.{silver_table}")
)

print(f"Wrote {silver_lakehouse}.{silver_table}")


## Validate silver output


In [ ]:
validation_df = spark.sql(f"""
SELECT
    COUNT(*) AS SilverRowCount,
    COUNT(DISTINCT TransactionID) AS DistinctTransactionIDs,
    SUM(CASE WHEN CardToken IS NULL THEN 1 ELSE 0 END) AS MissingCardTokens,
    SUM(CASE WHEN Amount IS NULL THEN 1 ELSE 0 END) AS MissingAmounts
FROM {silver_lakehouse}.{silver_table}
""")

display(validation_df)

display(spark.sql(f"""
SELECT AuthStatus, COUNT(*) AS TransactionCount, SUM(Amount) AS TotalAmount
FROM {silver_lakehouse}.{silver_table}
GROUP BY AuthStatus
ORDER BY TransactionCount DESC
"""))
